In [2]:
import os
import requests
import pandas as pd
import time
from tqdm import tqdm
from datetime import datetime, timedelta
import ssl
import warnings
from requests.packages.urllib3.exceptions import InsecureRequestWarning
from dotenv import load_dotenv
import json
import xml.etree.ElementTree as ET  # 추가: XML 파싱

load_dotenv()

True

In [ ]:
import tensorflow as tf
print(tf.__version__)
tf.config.list_physical_devices('GPU')

2.10.0


[PhysicalDevice(name='/physical_device:GPU:0', device_type='GPU')]

In [ ]:
# SSL 및 경고 설정
warnings.filterwarnings('ignore', category=InsecureRequestWarning)
ssl._create_default_https_context = ssl._create_unverified_context

In [ ]:
# API 설정
API_KEY = os.getenv('DO_API_KEY')  # 환경 변수에서 키를 불러옵니다
BASE_URL = 'http://apis.data.go.kr/B552845/katSale/trades'

In [ ]:
# 도매시장 코드 불러오기
df_market = pd.read_csv('도매시장_코드.csv', encoding='cp949')

In [ ]:
# 품목 코드 설정
ITEM_CODES = {
    "양파": "1201",  # 용곤
}

# 날짜 입력 (YYYY-MM-DD)  # 테스트 후에 일자 조정 =>
start_date = '2018-01-03'
end_date = '2020-12-31'
start_dt = datetime.strptime(start_date, '%Y-%m-%d')
end_dt = datetime.strptime(end_date, '%Y-%m-%d')
total_days = (end_dt - start_dt).days + 1

# 기타 설정
max_retries = 3
FAIL_LOG = []

In [ ]:
# 디렉토리 준비
os.makedirs("logs", exist_ok=True)
os.makedirs("data", exist_ok=True)

In [4]:
fail_df = pd.read_csv("data/무/유통공사_fail_log_무_20200101-20250531.csv", encoding="cp949")
fail_df.shape

(19352, 5)

In [5]:
fail_df

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 19352 entries, 0 to 19351
Data columns (total 5 columns):
 #   Column  Non-Null Count  Dtype 
---  ------  --------------  ----- 
 0   item    19352 non-null  object
 1   market  19352 non-null  object
 2   mcode   19352 non-null  int64 
 3   date    19352 non-null  object
 4   reason  19352 non-null  object
dtypes: int64(1), object(4)
memory usage: 756.1+ KB


In [22]:
df_date = pd.read_csv("holiday_score_table.csv", encoding="cp949")
df_holiday = df_date[df_date['holiday_flag']==1]
holidays = df_holiday['date'].values

In [24]:
holidays

array(['2018-01-01', '2018-01-06', '2018-01-07', '2018-01-13',
       '2018-01-14', '2018-01-20', '2018-01-21', '2018-01-27',
       '2018-01-28', '2018-02-03', '2018-02-04', '2018-02-10',
       '2018-02-11', '2018-02-15', '2018-02-16', '2018-02-17',
       '2018-02-18', '2018-02-24', '2018-02-25', '2018-03-01',
       '2018-03-03', '2018-03-04', '2018-03-10', '2018-03-11',
       '2018-03-17', '2018-03-18', '2018-03-24', '2018-03-25',
       '2018-03-31', '2018-04-01', '2018-04-07', '2018-04-08',
       '2018-04-14', '2018-04-15', '2018-04-21', '2018-04-22',
       '2018-04-28', '2018-04-29', '2018-05-05', '2018-05-06',
       '2018-05-07', '2018-05-12', '2018-05-13', '2018-05-19',
       '2018-05-20', '2018-05-22', '2018-05-26', '2018-05-27',
       '2018-06-02', '2018-06-03', '2018-06-06', '2018-06-09',
       '2018-06-10', '2018-06-13', '2018-06-16', '2018-06-17',
       '2018-06-23', '2018-06-24', '2018-06-30', '2018-07-01',
       '2018-07-07', '2018-07-08', '2018-07-14', '2018-

In [23]:
fail_df = pd.read_csv("data/무/유통공사_fail_log_무_20200101-20250531.csv", encoding="cp949")
fail_pairs = fail_df[['mcode', 'date']].drop_duplicates()
fail_pairs['mcode'] = fail_pairs['mcode'].astype(str)
for _, row in fail_pairs.iterrows():
    date_str = row['date']
    if date_str in holidays:
        print(date_str)

2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-01
2020-01-24
2020-01-24
2020-01-24
2020-01-24
2020-01-24
2020-01-24
2020-01-24
2020-01-24
2020-01-24
2020-01-24
2020-01-24
2020-01-24
2020-01-24
2020-01-24
2020-01-24
2020-01-24
2020-01-24
2020-01-24
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-01-27
2020-04-30
2020-04-30
2020-08-17
2020-09-30
2020-09-30
2020-09-30
2020-09-30
2020-09-30
2020-09-30
2020-09-30
2020-09-30

2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-23
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-01-24
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01
2023-03-01

In [26]:
import os
import pandas as pd
import requests
import time
from datetime import datetime
from tqdm import tqdm
import ssl
import warnings
import xml.etree.ElementTree as ET
from requests.packages.urllib3.exceptions import InsecureRequestWarning
from dotenv import load_dotenv

# 환경 및 경고 설정
load_dotenv()
warnings.filterwarnings('ignore', category=InsecureRequestWarning)
ssl._create_default_https_context = ssl._create_unverified_context

# 상수
API_KEY = os.getenv("DO_API_KEY")
BASE_URL = 'http://apis.data.go.kr/B552845/katSale/trades'
ITEM_CODES = {"무" : '1101'}
max_retries = 2  # 재시도 최대 횟수 (1회 시도 + 0회 재시도)

# 디렉토리 준비
os.makedirs("logs", exist_ok=True)
os.makedirs("data", exist_ok=True)
os.makedirs("success", exist_ok=True)

# 도매시장 코드 불러오기
df_market = pd.read_csv("도매시장_코드.csv", encoding="cp949", header=None)
df_market[0] = df_market[0].astype(str)

# 실패 로그 불러오기
fail_df = pd.read_csv("data/무/유통공사_fail_log_무_20200101-20250531.csv", encoding="cp949")
fail_pairs = fail_df[['mcode', 'date']].drop_duplicates()
fail_pairs['mcode'] = fail_pairs['mcode'].astype(str)

# 휴일 일자 불러오기
df_date = pd.read_csv("holiday_score_table.csv", encoding="cp949")
df_holiday = df_date[df_date['holiday_flag']==1]
holidays = df_holiday['date'].values

for item_name, code in ITEM_CODES.items():
    LARGE = code[:2]
    MID = code[2:]
    data_list = []
    cnt=0

    print(f"\n📦 실패 항목 재시도 시작: {item_name}")
    for _, row in tqdm(fail_pairs.iterrows(), total=len(fail_pairs), desc="재시도 진행"):
        mcode = str(row['mcode'])
        if mcode in (210005, 370401):
            continue
        date_str = row['date']
        if date_str in holidays:
            continue        

        market_name_row = df_market[df_market[0] == mcode]
        if market_name_row.empty:
            print(f"❌ 시장 코드 {mcode} 누락 - 스킵")
            continue
        market_name = market_name_row.values[0][1]

        retry_count = 0
        market_success = False

        while retry_count < max_retries:
            page_no = 1
            try:
                while True:
                    print(f"▶️ 요청 시도: {item_name} | 시장코드: {mcode} | 날짜: {date_str} | 페이지: {page_no} | 재시도: {retry_count + 1}")

                    params = {
                        'serviceKey': API_KEY,
                        'pageNo': page_no,
                        'numOfRows': 100,
                        'cond[trd_clcln_ymd::EQ]': date_str,
                        'cond[whsl_mrkt_cd::EQ]': mcode,
                        'cond[gds_lclsf_cd::EQ]': LARGE,
                        'cond[gds_mclsf_cd::EQ]': MID
                    }

                    response = requests.get(BASE_URL, params=params, verify=False, timeout=10)
                    content_type = response.headers.get("Content-Type", "")
                    time.sleep(1.0)
                    response_preview = response.text[:500].strip()

                    # 에러 체크
                    if "LIMITED_" in response_preview:
                        fail_reason = "❌ API 호출 제한 (LIMITED_ 응답)"
                    elif "SERVICE ERROR" in response_preview:
                        fail_reason = "❌ 서비스 오류 (SERVICE ERROR 응답)"
                    elif "ERROR" in response_preview.upper():
                        fail_reason = "❌ 기타 오류 포함 (ERROR 키워드 포함)"
                    elif "TOO MANY REQUESTS" in response_preview.upper():
                        fail_reason = "❌ 요청 과다로 인한 제한 (Too Many Requests)"
                    else:
                        fail_reason = None

                    if fail_reason:
                        print(f"⛔ {fail_reason} - 재시도 대기 중 (2분)")
                        log_prefix = f"logs/retry_failed_{item_name}_{mcode}_{date_str}"
                        with open(f"{log_prefix}.html", "w", encoding="utf-8") as f:
                            f.write(response.text)
                        with open(f"{log_prefix}_info.txt", "w", encoding="utf-8") as f:
                            f.write(f"[오류] {fail_reason}\n{response_preview}")
                        retry_count += 1
                        if retry_count >= max_retries:
                            print(f"❗ 최대 재시도 {max_retries}회 초과 - 중단")
                            break
                        time.sleep(60)
                        continue

                    # 응답 파싱
                    if "application/json" in content_type:
                        json_data = response.json()
                        body = json_data.get("response", {}).get("body", {})
                        items = body.get("items", {}).get("item", [])
                        total_count = int(body.get("totalCount", 0))

                    elif "application/xml" in content_type or response.text.strip().startswith("<"):
                        root = ET.fromstring(response.text)
                        total_count_el = root.find(".//totalCount")
                        total_count = int(total_count_el.text) if total_count_el is not None else 0
                        item_els = root.findall(".//item")
                        items = [{el.tag: el.text for el in item} for item in item_els]

                    else:
                        raise ValueError(f"알 수 없는 응답 형식: {content_type}")

                    if not items:
                        print("⚠️ 거래 데이터 없음")
                        market_success = True
                        break

                    data_list.extend(items)
                    cnt += 1

                    if cnt%5000 == 0:
                        print(f"🧪 중간 저장 시도: 현재 data_list 길이 = {len(data_list)}")
                        mid_save_path = f"data/retry/유통공사_retry_{item_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}_mid.csv"
                        df_mid = pd.DataFrame(data_list)
                        df_mid.to_csv(mid_save_path, encoding='cp949', index=False)
                        print(f"💾 중간 저장 완료: {mid_save_path}")
                        time.sleep(0.1)

                    market_success = True
                    if page_no * 100 >= total_count:
                        print(f"✅ 마지막 페이지 도달 (totalCount: {total_count})")
                        break
                    if page_no > 10:
                        print("🚨 페이지 10 초과 - 무한 루프 방지를 위해 중단")
                        break

                    page_no += 1
                    time.sleep(1.0)

                if market_success:
                    break
                else:
                    retry_count += 1
                    if retry_count >= max_retries:
                        print(f"❗ 최대 재시도 {max_retries}회 초과 - 중단")
                        break
                    time.sleep(2 * retry_count)

            except Exception as e:
                retry_count += 1
                print(f"❗예외 발생: {e} (재시도 {retry_count}/{max_retries})")
                fail_log_prefix = f"logs/retry_failed_{item_name}_{mcode}_{date_str}_try{retry_count}"
                if 'response' in locals():
                    with open(f"{fail_log_prefix}.txt", "w", encoding="utf-8") as f:
                        f.write(response.text)
                with open(f"{fail_log_prefix}_info.txt", "w", encoding="utf-8") as f:
                    f.write(f"[예외] {str(e)}\n")
                if retry_count >= max_retries:
                    print(f"❗ 최대 재시도 {max_retries}회 초과 - 중단")
                    break
                time.sleep(2 * retry_count)

        if not market_success:
            fail_log_path = f"data/logs/retry_failed_{item_name}_{mcode}_{date_str}.txt"
            with open(fail_log_path, "w", encoding="utf-8") as f:
                f.write(f"❌ {datetime.now()} - {item_name} {mcode} {date_str} 데이터 수집 실패\n")

    # DataFrame 생성 전 타입 검사
    if data_list:
        if not all(isinstance(item, dict) for item in data_list):
            raise ValueError("data_list에는 dict가 아닌 항목이 있습니다.")

        df = pd.DataFrame(data_list)
        filename = f"data/유통공사_retry_{item_name}_{datetime.now().strftime('%Y%m%d_%H%M%S')}.csv"
        df.to_csv(filename, encoding='cp949', index=False)
        print(f"✅ 저장 완료: {filename}")
    else:
        print(f"⚠️ {item_name}: 재시도에서도 데이터 없음")



📦 실패 항목 재시도 시작: 무


재시도 진행:   0%|                                                                           | 0/19352 [00:00<?, ?it/s]

▶️ 요청 시도: 무 | 시장코드: 110008 | 날짜: 2020-01-02 | 페이지: 1 | 재시도: 1
⛔ ❌ API 호출 제한 (LIMITED_ 응답) - 재시도 대기 중 (2분)
▶️ 요청 시도: 무 | 시장코드: 110008 | 날짜: 2020-01-02 | 페이지: 1 | 재시도: 2


재시도 진행:   0%|                                                               | 32/19352 [01:02<10:24:57,  1.94s/it]

⛔ ❌ API 호출 제한 (LIMITED_ 응답) - 재시도 대기 중 (2분)
❗ 최대 재시도 2회 초과 - 중단
❗ 최대 재시도 2회 초과 - 중단
▶️ 요청 시도: 무 | 시장코드: 210001 | 날짜: 2020-01-02 | 페이지: 1 | 재시도: 1
⛔ ❌ API 호출 제한 (LIMITED_ 응답) - 재시도 대기 중 (2분)


재시도 진행:   0%|                                                               | 32/19352 [02:03<20:39:46,  3.85s/it]


KeyboardInterrupt: 

In [ ]:
# 형식변환

import pandas as pd
import ast

# 파일 경로
retry_success_path = "유통공사_retry_success_20250708_084350.csv"
template_path = "유통공사_도매시장_배추_20180103-20241231.csv"
output_path = "유통공사_retry_success_형식변환_20250708.csv"

# CSV 파일 불러오기 (한글 포함이므로 cp949 인코딩)
df_retry_success = pd.read_csv(retry_success_path, encoding='cp949')
df_template = pd.read_csv(template_path, encoding='cp949')

# "0" 값만 있는 행 제거
df_filtered = df_retry_success[df_retry_success["0"] != "0"].copy()
df_filtered.columns = ["raw"]  # 열 이름 재지정

# 문자열을 파이썬 딕셔너리로 안전하게 변환
df_parsed = df_filtered["raw"].apply(ast.literal_eval)
df_data = pd.DataFrame(df_parsed.tolist())

# 열 순서 맞추기 (template 기준)
df_data = df_data.reindex(columns=df_template.columns)

# 저장
df_data.to_csv(output_path, index=False, encoding='cp949')

In [7]:
len(data_list)

109

In [8]:
data_list

[{'trd_clcln_ymd': '2018-01-03',
  'whsl_mrkt_cd': '210005',
  'gds_cd': '1201',
  'note': '거래 없음'},
 {'trd_clcln_ymd': '2018-01-03',
  'whsl_mrkt_cd': '370401',
  'gds_cd': '1201',
  'note': '거래 없음'},
 {'trd_clcln_ymd': '2018-01-04',
  'whsl_mrkt_cd': '210005',
  'gds_cd': '1201',
  'note': '거래 없음'},
 {'trd_clcln_ymd': '2018-01-04',
  'whsl_mrkt_cd': '370401',
  'gds_cd': '1201',
  'note': '거래 없음'},
 {'trd_clcln_ymd': '2018-01-05',
  'whsl_mrkt_cd': '210005',
  'gds_cd': '1201',
  'note': '거래 없음'},
 {'trd_clcln_ymd': '2018-01-05',
  'whsl_mrkt_cd': '370401',
  'gds_cd': '1201',
  'note': '거래 없음'},
 {'trd_clcln_ymd': '2018-01-06',
  'whsl_mrkt_cd': '210005',
  'gds_cd': '1201',
  'note': '거래 없음'},
 {'trd_clcln_ymd': '2018-01-06',
  'whsl_mrkt_cd': '320101',
  'gds_cd': '1201',
  'note': '거래 없음'},
 {'trd_clcln_ymd': '2018-01-06',
  'whsl_mrkt_cd': '320201',
  'gds_cd': '1201',
  'note': '거래 없음'},
 {'trd_clcln_ymd': '2018-01-06',
  'whsl_mrkt_cd': '370401',
  'gds_cd': '1201',
  'note': 